# S50_04 — Cost Optimization

LLM API costs scale with token volume. A naive implementation of a chat application can easily spend $10k/month on what should cost $100. This notebook covers the main levers for reducing LLM costs.

## Lever 1: Choose the right model

In [ ]:
import anthropic
import time

client = anthropic.Anthropic()

# Model pricing comparison (per million tokens, as of 2026)
models = {
    'claude-haiku-4-5-20251001':  {'input': 1.00,  'output': 5.00,   'quality': 'Good'},
    'claude-sonnet-4-6':          {'input': 3.00,  'output': 15.00,  'quality': 'Great'},
    'claude-opus-4-7':            {'input': 15.00, 'output': 75.00,  'quality': 'Best'},
    'gpt-4o-mini':                {'input': 0.15,  'output': 0.60,   'quality': 'Good'},
    'gpt-4o':                     {'input': 2.50,  'output': 10.00,  'quality': 'Great'},
}

# Cost for 1M requests with 500 input + 200 output tokens each
input_tokens = 500
output_tokens = 200
n_requests = 1_000_000

print(f'Cost for {n_requests:,} requests ({input_tokens} in / {output_tokens} out tokens each):')
print(f'{"Model":30} {"$/request":>12} {"Monthly cost":>15} {"Quality"}')
print('-' * 70)
for model, info in models.items():
    cost_per_req = (input_tokens * info['input'] + output_tokens * info['output']) / 1e6
    monthly = cost_per_req * n_requests
    print(f'{model:30} ${cost_per_req:>10.5f} ${monthly:>14,.0f}   {info["quality"]}')

## Lever 2: Prompt caching

In [ ]:
# Prompt caching: re-use a large system prompt / document prefix
# Cost of cached tokens: ~10% of normal input price

# Scenario: RAG with a 5000-token system prompt repeated for every user question
system_tokens = 5000
user_tokens = 100
output_tokens = 200
n_requests = 100_000

haiku_price = {'input': 1.00/1e6, 'output': 5.00/1e6, 'cache_write': 1.25/1e6, 'cache_read': 0.10/1e6}

# Without caching
no_cache_cost = n_requests * (
    (system_tokens + user_tokens) * haiku_price['input'] +
    output_tokens * haiku_price['output']
)

# With caching (system prompt cached after first request)
cache_cost = (
    system_tokens * haiku_price['cache_write'] +              # first write
    (n_requests - 1) * system_tokens * haiku_price['cache_read'] +  # subsequent reads
    n_requests * user_tokens * haiku_price['input'] +
    n_requests * output_tokens * haiku_price['output']
)

print(f'Scenario: {n_requests:,} requests, {system_tokens}-token system prompt')
print(f'Without caching: ${no_cache_cost:.2f}')
print(f'With caching:    ${cache_cost:.2f}')
print(f'Savings:         ${no_cache_cost - cache_cost:.2f} ({(1 - cache_cost/no_cache_cost)*100:.0f}%)')

## Lever 3: Output length control

In [ ]:
# Output tokens are 5–15x more expensive than input tokens
# Constrain output length whenever possible

# Bad: open-ended → model writes a lot
verbose = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=512,
    messages=[{'role': 'user', 'content': 'What is a transformer?'}],
)

# Good: constrain format → much shorter
concise = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=512,
    messages=[{'role': 'user', 'content': 'What is a transformer? Answer in exactly 2 sentences.'}],
)

print(f'Verbose output tokens:  {verbose.usage.output_tokens}')
print(f'Concise output tokens:  {concise.usage.output_tokens}')

haiku_out_price = 5.00 / 1e6
saving = (verbose.usage.output_tokens - concise.usage.output_tokens) * haiku_out_price
print(f'Saving per request: ${saving:.6f}')
print(f'At 1M requests/month: ${saving * 1e6:.2f}/month')

## Lever 4: Caching at the application level

In [ ]:
import hashlib
import json

class CachedLLMClient:
    """In-memory semantic cache — skip the LLM for repeated prompts."""
    
    def __init__(self):
        self.client = anthropic.Anthropic()
        self.cache = {}  # in production: Redis or similar
        self.hits = 0
        self.misses = 0
    
    def _cache_key(self, messages, system=''):
        payload = json.dumps({'system': system, 'messages': messages}, sort_keys=True)
        return hashlib.sha256(payload.encode()).hexdigest()
    
    def ask(self, messages, system='', model='claude-haiku-4-5-20251001', max_tokens=256):
        key = self._cache_key(messages, system)
        
        if key in self.cache:
            self.hits += 1
            return self.cache[key], True  # (response, from_cache)
        
        self.misses += 1
        kwargs = dict(model=model, max_tokens=max_tokens, messages=messages)
        if system:
            kwargs['system'] = system
        result = self.client.messages.create(**kwargs)
        response = result.content[0].text
        self.cache[key] = response
        return response, False

cached_client = CachedLLMClient()

q = [{'role': 'user', 'content': 'What is overfitting?'}]
r1, from_cache1 = cached_client.ask(q)
r2, from_cache2 = cached_client.ask(q)  # same prompt
r3, from_cache3 = cached_client.ask([{'role': 'user', 'content': 'What is underfitting?'}])

print(f'Hit 1 (new):    from_cache={from_cache1}')
print(f'Hit 2 (repeat): from_cache={from_cache2}')
print(f'Hit 3 (new):    from_cache={from_cache3}')
print(f'Cache hit rate: {cached_client.hits/(cached_client.hits+cached_client.misses)*100:.0f}%')

## Cost optimization checklist

- [ ] Use the cheapest model that meets quality requirements (Haiku for simple tasks)
- [ ] Enable prompt caching for long system prompts (saves up to 90% on system prompt tokens)
- [ ] Constrain output length with format instructions
- [ ] Cache responses for repeated identical prompts
- [ ] Batch requests when real-time response is not needed
- [ ] Use local models (Ollama) for development to avoid API costs
- [ ] Set `max_tokens` to just above what you expect — prevents accidentally long outputs
- [ ] Monitor cost per user/session to catch runaway usage early

Next: [S50_05_llmops_overview.ipynb](./S50_05_llmops_overview.ipynb)